# 6.1 PyTorch → ONNX — Deep Dive

## Table of Contents
1. [The Export Pipeline](#section-1)
2. [Tracing Semantics](#section-2)
3. [TorchScript IR Lowering to ONNX](#section-3)
4. [Dynamic Shape Algebra](#section-4)
5. [`torch.onnx.export()` API Reference](#section-5)
6. [Tracing vs Scripting — Decision Framework](#section-6)
7. [Constant Folding Optimization](#section-7)
8. [Building and Exporting a CNN Model](#section-8)
9. [Numerical Parity Checking](#section-9)
10. [Common Pitfalls and Mitigations](#section-10)
11. [Dynamic Axes for NLP Models](#section-11)
12. [Key Takeaways](#section-12)

<a id='section-1'></a>
## Section 1: The Export Pipeline

Exporting a PyTorch model to ONNX is **not** a simple serialization of Python
objects. It is a multi-stage compiler pipeline that transforms a dynamic,
eagerly-evaluated `nn.Module` into a static, platform-independent computation
graph.

```
┌──────────────────────────────────────────────────────────────────────────┐
│                   PyTorch → ONNX EXPORT PIPELINE                        │
├──────────────────────────────────────────────────────────────────────────┤
│                                                                          │
│   nn.Module          TorchScript IR        ONNX Lowering     .onnx file │
│  ┌──────────┐       ┌──────────────┐      ┌────────────┐   ┌─────────┐ │
│  │ Python   │ trace │  %x = aten:: │ lower│  NodeProto  │   │ProtoBuf│ │
│  │ forward()│──────▶│  conv2d(...)  │─────▶│  op="Conv"  │──▶│ binary  │ │
│  │ method   │  or   │  %y = aten:: │      │  op="Relu"  │   │ file    │ │
│  │          │script │  relu(...)    │      │  op="Add"   │   │         │ │
│  └──────────┘       └──────────────┘      └────────────┘   └─────────┘ │
│       │                    │                     │               │       │
│       ▼                    ▼                     ▼               ▼       │
│  Eager Python        Static graph          ONNX standard    Portable    │
│  with control        with typed SSA        operator set     artifact    │
│  flow, dynamic       form nodes            (opset N)                    │
│  dispatch                                                               │
└──────────────────────────────────────────────────────────────────────────┘
```

The pipeline can be summarized mathematically. Given a module $M$ with
forward method $M.\text{forward}: \mathcal{X} \to \mathcal{Y}$ and a
representative input $x_0 \in \mathcal{X}$:

$$G_{\text{onnx}} = \text{lower}_{\text{ONNX}}\bigl(\text{capture}(M.\text{forward},\, x_0)\bigr)$$

where $\text{capture}$ produces the TorchScript IR and $\text{lower}_{\text{ONNX}}$
maps each TorchScript node to the corresponding ONNX operator(s).

![PyTorch to ONNX Export](assets/pytorch_to_onnx_export.png)

<a id='section-2'></a>
## Section 2: Tracing Semantics

### How Tracing Works

**Tracing** runs the model's `forward()` method with a concrete input tensor
and records every operator that executes. The result is a straight-line graph
with no branches or loops — a single execution path frozen as the model.

Formally, let $f = M.\text{forward}$ and let $x_0$ be the example input:

$$f_{\text{trace}}(x_0) = \text{record}\bigl(f(x_0)\bigr)$$

The traced graph is **valid for all inputs that follow the same execution path**
as $x_0$. If the model contains data-dependent control flow (e.g.,
`if x.sum() > 0`), only the branch taken during tracing is captured.

### What Tracing Captures vs Misses

```
┌─────────────────────────────────────────────────────────────────┐
│                    TRACING CAPTURE MODEL                        │
├───────────────────────────┬─────────────────────────────────────┤
│   CAPTURED (safe)         │   MISSED (dangerous)                │
├───────────────────────────┼─────────────────────────────────────┤
│ • torch.* tensor ops      │ • Python if/else on tensor values   │
│ • nn.Module submodules    │ • Python for loops (unrolled once)  │
│ • In-place operations     │ • .item(), .tolist() in control     │
│ • torch.nn.functional.*   │ • Data-dependent indexing           │
│ • Static control flow     │ • External side effects             │
│   (always same branch)    │ • Dynamic module selection          │
└───────────────────────────┴─────────────────────────────────────┘
```

### The Trace Validity Condition

A traced graph $G_{\text{trace}}$ produces correct output for input $x$ if and
only if $x$ exercises the **same operator sequence** as $x_0$:

$$G_{\text{trace}}(x) = f(x) \iff \text{path}(f, x) = \text{path}(f, x_0)$$

For models without data-dependent branching (most CNNs, transformers with
fixed architecture), this holds universally — the trace is always correct
regardless of input values.

In [ ]:
import torch
import torch.nn as nn

class BranchyModel(nn.Module):
    """Model with data-dependent control flow — tracing will miss a branch."""
    def __init__(self):
        super().__init__()
        self.linear_pos = nn.Linear(4, 4)
        self.linear_neg = nn.Linear(4, 4)

    def forward(self, x):
        if x.mean() > 0:              # data-dependent branch
            return self.linear_pos(x)
        else:
            return self.linear_neg(x)

model = BranchyModel()
model.eval()

positive_input = torch.ones(2, 4)
traced = torch.jit.trace(model, positive_input)

print("Traced graph (only positive branch captured):")
print(traced.graph)
print("\nWarning: negative-input branch is permanently lost in this trace.")

<a id='section-3'></a>
## Section 3: TorchScript IR Lowering to ONNX

### The Lowering Process

After tracing (or scripting), PyTorch holds an intermediate representation in
**TorchScript IR** — a typed, SSA (Static Single Assignment) graph. The ONNX
exporter then **lowers** each TorchScript node to one or more ONNX `NodeProto`
objects.

```
TorchScript IR Node                    ONNX NodeProto(s)
──────────────────                    ──────────────────
aten::conv2d(...)          ───────▶   op_type = "Conv"
                                      attributes: kernel_shape, strides, pads

aten::batch_norm(...)      ───────▶   op_type = "BatchNormalization"
                                      attributes: epsilon, momentum

aten::relu(...)            ───────▶   op_type = "Relu"

aten::addmm(bias, x, w)   ───────▶   op_type = "Gemm"  (or MatMul + Add)
                                      attributes: alpha, beta, transB

aten::flatten(x, 1, -1)   ───────▶   op_type = "Flatten"
                                      attributes: axis=1
```

### ATen Operator Decomposition

Not every ATen operator has a direct ONNX equivalent. Complex operators are
**decomposed** into sequences of primitive ONNX ops:

$$\text{aten::layer\_norm}(x, \gamma, \beta) \longrightarrow \begin{cases}
\mu = \text{ReduceMean}(x) \\
\sigma^2 = \text{ReduceMean}((x - \mu)^2) \\
\hat{x} = (x - \mu) / \sqrt{\sigma^2 + \epsilon} \\
y = \gamma \cdot \hat{x} + \beta
\end{cases}$$

This decomposition is opset-version–dependent. Higher opset versions introduce
dedicated operators (e.g., `LayerNormalization` was added in opset 17), reducing
graph complexity.

### The Mapping Function

The lowering can be described as a mapping $\phi$ from TorchScript nodes to
ONNX subgraphs:

$$\phi: \text{TSNode} \to \{\text{ONNXNode}\}^+$$

where $|\phi(n)| \geq 1$ — each TorchScript node maps to **at least one** ONNX
node. When $|\phi(n)| > 1$, the operator has been decomposed.

In [ ]:
import torch
import torch.nn as nn
import tempfile, os, onnx

class LayerNormExample(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln = nn.LayerNorm(16)
        self.fc = nn.Linear(16, 4)

    def forward(self, x):
        return self.fc(self.ln(x))

model = LayerNormExample()
model.eval()
dummy = torch.randn(2, 16)

for opset in [13, 17]:
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, f"ln_opset{opset}.onnx")
        torch.onnx.export(model, dummy, path, opset_version=opset,
                          input_names=["x"], output_names=["y"])
        m = onnx.load(path)
        ops = [n.op_type for n in m.graph.node]
        print(f"Opset {opset}: {len(ops)} nodes → {ops}")

<a id='section-4'></a>
## Section 4: Dynamic Shape Algebra

### The Problem with Fixed Shapes

By default, tracing records **concrete dimension values** from the example
input. If you trace with a batch size of 2, the exported graph expects exactly
batch size 2 at inference time. This is almost never what you want.

### The `dynamic_axes` Contract

The `dynamic_axes` parameter tells the exporter which dimensions should remain
**symbolic** rather than being baked in as constants:

$$\text{dynamic\_axes} = \{(\text{tensor\_name},\, \text{dim\_idx}): \text{symbolic\_name}\}$$

For example, making batch dimension dynamic:

$$\text{dynamic\_axes} = \{(\texttt{"input"},\, 0): \texttt{"batch"},\; (\texttt{"output"},\, 0): \texttt{"batch"}\}$$

### Shape Constraint Propagation

When a dimension is marked dynamic, the ONNX shape inference engine propagates
symbolic dimensions through the graph:

```
                        Shape Propagation Example
                        ─────────────────────────

  Input: [batch, 1, 28, 28]     ← batch is symbolic
     │
     ▼
  Conv2d(1→16, 3×3, pad=1)
  Output: [batch, 16, 28, 28]   ← batch propagated
     │
     ▼
  MaxPool2d(2×2)
  Output: [batch, 16, 14, 14]   ← batch propagated
     │
     ▼
  Flatten(start_dim=1)
  Output: [batch, 3136]         ← batch propagated
     │
     ▼
  Linear(3136→10)
  Output: [batch, 10]           ← batch propagated
```

### Without dynamic_axes

If you omit `dynamic_axes`, the shape of every tensor is **fixed**:

$$\text{shape}(\texttt{input}) = [2, 1, 28, 28] \quad \text{(frozen constant)}$$

Some runtimes will still accept different batch sizes (ONNX Runtime is lenient),
but others (TensorRT, CoreML) will reject or mishandle inputs that don't match
the declared static shapes.

In [ ]:
import torch
import torch.nn as nn
import tempfile, os, onnx

model = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 4),
)
model.eval()
dummy = torch.randn(3, 8)

with tempfile.TemporaryDirectory() as tmp:
    static_path = os.path.join(tmp, "static.onnx")
    torch.onnx.export(model, dummy, static_path, opset_version=17,
                      input_names=["x"], output_names=["y"])
    m_static = onnx.load(static_path)

    dynamic_path = os.path.join(tmp, "dynamic.onnx")
    torch.onnx.export(model, dummy, dynamic_path, opset_version=17,
                      input_names=["x"], output_names=["y"],
                      dynamic_axes={"x": {0: "batch"}, "y": {0: "batch"}})
    m_dynamic = onnx.load(dynamic_path)

def show_io_shapes(model_proto, label):
    print(f"\n{label}:")
    for inp in model_proto.graph.input:
        dims = [d.dim_param or str(d.dim_value)
                for d in inp.type.tensor_type.shape.dim]
        print(f"  Input '{inp.name}': [{', '.join(dims)}]")
    for out in model_proto.graph.output:
        dims = [d.dim_param or str(d.dim_value)
                for d in out.type.tensor_type.shape.dim]
        print(f"  Output '{out.name}': [{', '.join(dims)}]")

show_io_shapes(m_static, "WITHOUT dynamic_axes")
show_io_shapes(m_dynamic, "WITH dynamic_axes")

<a id='section-5'></a>
## Section 5: `torch.onnx.export()` API Reference

The main entry point for PyTorch→ONNX export. Below is a comprehensive
parameter reference with semantic explanations.

```
torch.onnx.export(
    model,                    # nn.Module — MUST be in eval() mode
    args,                     # tuple of example inputs for forward()
    f,                        # output file path or file-like object
    export_params=True,       # embed weights in the .onnx file
    verbose=False,            # print TorchScript graph during export
    training=TrainingMode.EVAL,  # export in eval mode
    input_names=None,         # human-readable input tensor names
    output_names=None,        # human-readable output tensor names
    opset_version=None,       # target ONNX opset version
    dynamic_axes=None,        # {name: {dim_idx: symbolic_name}}
    do_constant_folding=True, # pre-compute constant subgraphs
    keep_initializers_as_inputs=None,
    operator_export_type=OperatorExportTypes.ONNX,
)
```

### Parameter Semantics Table

| Parameter | Type | Purpose | Recommendation |
|-----------|------|---------|----------------|
| `model` | `nn.Module` | The PyTorch model to export | Always call `.eval()` first |
| `args` | `tuple` or `Tensor` | Concrete example inputs that drive tracing | Match production dtype and rank |
| `f` | `str` or `IO` | Output file path for the `.onnx` artifact | Use `.onnx` extension |
| `export_params` | `bool` | If True, embed learned parameters in the file | Keep True (default) |
| `opset_version` | `int` | Target ONNX opset (default: latest PyTorch knows) | Use highest opset your runtime supports |
| `input_names` | `list[str]` | Names for graph input tensors | Always set for clarity |
| `output_names` | `list[str]` | Names for graph output tensors | Always set for clarity |
| `dynamic_axes` | `dict` | Map of tensor names to dynamic dimension indices | Set for batch, seq_len dimensions |
| `do_constant_folding` | `bool` | Pre-evaluate constant subexpressions | Keep True (default) |

### Opset Version Selection

The opset version determines which ONNX operators are available. Higher opsets
provide more expressive operators and reduce graph decomposition:

$$\text{graph\_complexity}(\text{opset}) \propto \frac{1}{\text{opset\_version}}$$

| Opset | Key Additions |
|-------|---------------|
| 11 | Resize, ScatterND, dynamic shapes improved |
| 13 | Squeeze/Unsqueeze take axis as input, Split improvements |
| 15 | Shape + BatchNormalization updates |
| 17 | LayerNormalization, GroupNormalization |
| 18 | BitwiseAnd/Or/Xor, Pad improvements |

<a id='section-6'></a>
## Section 6: Tracing vs Scripting — Decision Framework

PyTorch offers two capture mechanisms for building the TorchScript IR:
**tracing** and **scripting**. They serve different model architectures.

### Tracing

- Runs `forward()` with a real tensor, records every op
- Produces a straight-line graph (no control flow)
- Fast, simple, works for most production models
- Silently drops data-dependent branches

### Scripting

- Statically analyzes Python source code
- Preserves `if`, `for`, `while` as TorchScript IR constructs
- Requires Python subset compliance (no arbitrary Python)
- More verbose errors, stricter type requirements

### Decision Tree

```
                    Does your model have
                  data-dependent control flow?
                  ┌──────────┴──────────┐
                  │                     │
                 NO                    YES
                  │                     │
                  ▼                     ▼
            ┌──────────┐       Can it be rewritten
            │  TRACE   │       with torch.where /
            │ (simple, │       masked ops?
            │  fast)   │       ┌─────┴─────┐
            └──────────┘      YES          NO
                               │            │
                               ▼            ▼
                         ┌──────────┐ ┌──────────┐
                         │ Rewrite  │ │  SCRIPT  │
                         │ then     │ │ (preserves│
                         │ TRACE    │ │ control   │
                         └──────────┘ │ flow)     │
                                      └──────────┘
```

### Comparison Table

| Aspect | Tracing | Scripting |
|--------|---------|----------|
| Control flow | Frozen (one path) | Preserved |
| Python support | Full (but only traces ops) | Restricted subset |
| Typical use | CNNs, Transformers, fixed-arch | RNNs with early stop, dynamic routing |
| Error messages | Silent wrongness possible | Explicit type errors |
| Graph quality | Clean, optimizable | May have extra control nodes |

In [ ]:
import torch
import torch.nn as nn

class MaskedModel(nn.Module):
    """Rewrite branching with torch.where for trace-safe export."""
    def __init__(self):
        super().__init__()
        self.linear_pos = nn.Linear(4, 4)
        self.linear_neg = nn.Linear(4, 4)

    def forward(self, x):
        pos_out = self.linear_pos(x)
        neg_out = self.linear_neg(x)
        mask = (x.mean(dim=-1, keepdim=True) > 0).float()
        return mask * pos_out + (1 - mask) * neg_out

model = MaskedModel()
model.eval()

traced = torch.jit.trace(model, torch.randn(2, 4))
print("Trace-safe graph (both paths always execute):")
print(traced.graph)

pos_x = torch.ones(2, 4)
neg_x = -torch.ones(2, 4)
print(f"\nPositive input result: {traced(pos_x).detach().numpy().round(3)}")
print(f"Negative input result: {traced(neg_x).detach().numpy().round(3)}")

<a id='section-7'></a>
## Section 7: Constant Folding Optimization

When `do_constant_folding=True` (the default), the exporter evaluates
subexpressions whose inputs are all known at export time and replaces
them with their computed values.

### How It Works

Consider a `BatchNorm` layer in eval mode. Its running mean $\mu$,
running variance $\sigma^2$, weight $\gamma$, and bias $\beta$ are all
constants at inference time. The normalization:

$$y = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

can be pre-computed into a simple affine transform:

$$y = \underbrace{\frac{\gamma}{\sqrt{\sigma^2 + \epsilon}}}_{w_{\text{folded}}} \cdot x + \underbrace{\beta - \frac{\gamma \cdot \mu}{\sqrt{\sigma^2 + \epsilon}}}_{b_{\text{folded}}}$$

```
Before Constant Folding:           After Constant Folding:
───────────────────────           ──────────────────────

  x ──▶ [BatchNorm] ──▶ y          x ──▶ [Mul(w_f)] ──▶ [Add(b_f)] ──▶ y
          │  │  │  │
          μ  σ² γ  β                (μ, σ², γ, β folded into w_f, b_f)
        (4 constant inputs)         (2 constant tensors, fewer ops)
```

### Impact on Graph Size

For models with many BatchNorm layers (e.g., ResNet-50 has 53 BN layers),
constant folding can significantly reduce node count and file size.

In [ ]:
import torch
import torch.nn as nn
import tempfile, os, onnx

class BNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 16, 3, padding=1)
        self.bn = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

model = BNModel()
model.eval()
dummy = torch.randn(1, 3, 8, 8)

with tempfile.TemporaryDirectory() as tmp:
    for fold in [False, True]:
        path = os.path.join(tmp, f"bn_fold_{fold}.onnx")
        torch.onnx.export(model, dummy, path, opset_version=17,
                          input_names=["x"], output_names=["y"],
                          do_constant_folding=fold)
        m = onnx.load(path)
        ops = [n.op_type for n in m.graph.node]
        size = os.path.getsize(path)
        print(f"constant_folding={fold}: {len(ops)} nodes, {size:,} bytes")
        print(f"  Ops: {ops}")

![ONNX Export Pipeline](assets/onnx_export_pipeline.png)

<a id='section-8'></a>
## Section 8: Building and Exporting a CNN Model

Let's build a complete CNN for MNIST-style images and walk through the full
export process with best practices applied.

### Model Architecture

```
Input: [batch, 1, 28, 28]
  │
  ├─▶ Conv2d(1→16, 3×3, pad=1)  →  [batch, 16, 28, 28]
  ├─▶ BatchNorm2d(16)            →  [batch, 16, 28, 28]
  ├─▶ ReLU                       →  [batch, 16, 28, 28]
  ├─▶ MaxPool2d(2)               →  [batch, 16, 14, 14]
  │
  ├─▶ Conv2d(16→32, 3×3, pad=1) →  [batch, 32, 14, 14]
  ├─▶ BatchNorm2d(32)            →  [batch, 32, 14, 14]
  ├─▶ ReLU                       →  [batch, 32, 14, 14]
  ├─▶ MaxPool2d(2)               →  [batch, 32, 7, 7]
  │
  ├─▶ Flatten                    →  [batch, 1568]
  ├─▶ Linear(1568→128)           →  [batch, 128]
  ├─▶ ReLU                       →  [batch, 128]
  ├─▶ Dropout(0.3)               →  [batch, 128]  (identity in eval)
  └─▶ Linear(128→10)             →  [batch, 10]
```

In [ ]:
import os
import tempfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import onnx
from onnx import checker

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

model = SmallCNN()
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable:        {trainable:,}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "small_cnn.onnx")
    dummy = torch.randn(2, 1, 28, 28)

    torch.onnx.export(
        model,
        dummy,
        path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={
            "image":  {0: "batch_size"},
            "logits": {0: "batch_size"},
        },
    )

    onnx_model = onnx.load(path)
    checker.check_model(onnx_model)

    file_size = os.path.getsize(path)
    num_nodes = len(onnx_model.graph.node)
    unique_ops = sorted(set(n.op_type for n in onnx_model.graph.node))

    print(f"Export successful!")
    print(f"  File size:  {file_size:,} bytes ({file_size/1024:.1f} KB)")
    print(f"  Nodes:      {num_nodes}")
    print(f"  Unique ops: {unique_ops}")
    print(f"  Opset:      {onnx_model.opset_import[0].version}")

    print("\nGraph inputs:")
    for inp in onnx_model.graph.input:
        dims = [d.dim_param or str(d.dim_value)
                for d in inp.type.tensor_type.shape.dim]
        print(f"  {inp.name}: [{', '.join(dims)}]")
    print("Graph outputs:")
    for out in onnx_model.graph.output:
        dims = [d.dim_param or str(d.dim_value)
                for d in out.type.tensor_type.shape.dim]
        print(f"  {out.name}: [{', '.join(dims)}]")

<a id='section-9'></a>
## Section 9: Numerical Parity Checking

After export, you **must** verify that the ONNX model produces the same
outputs as the original PyTorch model. Small floating-point differences are
expected due to operator implementation differences, but large discrepancies
indicate an export bug.

### The Parity Test Protocol

1. Generate $N$ random test inputs matching production statistics
2. Run each through PyTorch (with `torch.no_grad()`)
3. Run each through ONNX Runtime
4. Compare outputs element-wise

### Tolerance Thresholds

For float32 models, the standard acceptance criteria:

$$\max_{i} |y^{\text{PT}}_i - y^{\text{ORT}}_i| < \epsilon_{\text{abs}} \quad \text{where } \epsilon_{\text{abs}} \approx 10^{-5}$$

For relative comparison (more robust when output magnitudes vary):

$$\max_{i} \frac{|y^{\text{PT}}_i - y^{\text{ORT}}_i|}{\max(|y^{\text{PT}}_i|, \epsilon)} < \epsilon_{\text{rel}} \quad \text{where } \epsilon_{\text{rel}} \approx 10^{-4}$$

In [ ]:
try:
    import onnxruntime as ort

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "cnn_parity.onnx")
        dummy = torch.randn(4, 1, 28, 28)
        torch.onnx.export(
            model, dummy, path,
            opset_version=17,
            input_names=["image"],
            output_names=["logits"],
            dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
            do_constant_folding=True,
        )

        sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])

        print("Parity test across multiple batch sizes:")
        print(f"{'Batch':>6} {'Max Abs Diff':>14} {'Mean Abs Diff':>14} {'Pass':>6}")
        print("-" * 44)

        for batch_size in [1, 4, 16, 32]:
            x_np = np.random.randn(batch_size, 1, 28, 28).astype(np.float32)

            with torch.no_grad():
                pt_out = model(torch.from_numpy(x_np)).numpy()

            ort_out = sess.run(None, {"image": x_np})[0]

            max_diff = np.abs(pt_out - ort_out).max()
            mean_diff = np.abs(pt_out - ort_out).mean()
            passed = max_diff < 1e-5

            print(f"{batch_size:>6} {max_diff:>14.2e} {mean_diff:>14.2e} {'OK' if passed else 'FAIL':>6}")

except ImportError:
    print("onnxruntime not installed — install with: pip install onnxruntime")

<a id='section-10'></a>
## Section 10: Common Pitfalls and Mitigations

The table below catalogs the most frequent export failures and how to fix them.

| # | Pitfall | Symptom | Root Cause | Mitigation |
|---|---------|---------|------------|------------|
| 1 | Forgot `model.eval()` | Non-deterministic outputs, BN uses batch stats | Dropout active, BN in train mode | **Always** call `model.eval()` before export |
| 2 | Wrong dummy input shape | Missing ops or wrong graph topology | Tracer follows shape-dependent code paths | Use inputs matching production dimensions |
| 3 | Python `for` loop in forward | Loop is unrolled with fixed iteration count | Tracer treats loop body as N sequential ops | Use `torch.jit.script` or vectorize |
| 4 | `.item()` / `.tolist()` | `TracerWarning` or export failure | Converts tensor to Python scalar, breaks graph | Keep all ops as tensor operations |
| 5 | Unsupported ATen op | `RuntimeError: ONNX export failed` | No registered ONNX mapping for the op | Upgrade opset, rewrite with supported ops, or register custom symbolic |
| 6 | Missing `dynamic_axes` | Model only works with traced batch size | Dimensions baked as constants | Always declare dynamic dims |
| 7 | `torch.tensor()` in forward | Creates constant tensor each call | Treated as initializer, not recomputed | Use `self.register_buffer()` instead |
| 8 | In-place ops on inputs | Potential numerical issues | ONNX assumes functional semantics | Avoid in-place ops on graph inputs |

### Custom Symbolic Registration

When an ATen operator lacks an ONNX mapping, you can register a custom
symbolic function:

```python
from torch.onnx import register_custom_op_symbolic

def my_custom_op_symbolic(g, input, param):
    return g.op("com.mydomain::MyOp", input, param_i=param)

register_custom_op_symbolic("aten::my_custom_op", my_custom_op_symbolic, 17)
```

In [ ]:
import torch
import torch.nn as nn

class PitfallDemo(nn.Module):
    """Demonstrates .item() pitfall — this will produce a tracer warning."""
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 4)

    def forward(self, x):
        scale = x.abs().mean().item()  # breaks trace graph
        return self.linear(x) * scale

class FixedModel(nn.Module):
    """Fixed version — keeps everything as tensor ops."""
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 4)

    def forward(self, x):
        scale = x.abs().mean()  # stays as tensor
        return self.linear(x) * scale

print("PitfallDemo uses .item() → will freeze the scalar as a constant.")
print("FixedModel keeps .mean() as tensor op → correct dynamic behavior.")

fixed = FixedModel()
fixed.eval()
import tempfile, os, onnx
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "fixed.onnx")
    torch.onnx.export(fixed, torch.randn(2, 4), path, opset_version=17,
                      input_names=["x"], output_names=["y"])
    m = onnx.load(path)
    print(f"\nFixed model exported: {len(m.graph.node)} nodes")
    print(f"Ops: {[n.op_type for n in m.graph.node]}")

<a id='section-11'></a>
## Section 11: Dynamic Axes for NLP Models

NLP models require dynamic axes on **two** dimensions: batch size and sequence
length. This is because sentences have variable lengths and are typically
padded to the longest sequence in each batch.

### Encoder-Only (BERT-like)

```python
dynamic_axes = {
    "input_ids":      {0: "batch", 1: "seq_len"},
    "attention_mask":  {0: "batch", 1: "seq_len"},
    "token_type_ids":  {0: "batch", 1: "seq_len"},
    "last_hidden_state": {0: "batch", 1: "seq_len"},
}
```

### Decoder-Only (GPT-like)

For autoregressive models with KV-cache, the cache dimensions also need
to be dynamic:

```python
dynamic_axes = {
    "input_ids":       {0: "batch", 1: "seq_len"},
    "past_key":        {0: "batch", 2: "past_len"},
    "past_value":      {0: "batch", 2: "past_len"},
    "logits":          {0: "batch", 1: "seq_len"},
}
```

### Shape Algebra for Attention

In a standard multi-head attention block, the key tensors have shape:

$$K \in \mathbb{R}^{B \times H \times S \times D_k}$$

where $B$ = batch, $H$ = heads, $S$ = sequence length, $D_k$ = head dimension.
The attention score matrix:

$$A = \text{softmax}\left(\frac{QK^T}{\sqrt{D_k}}\right) \in \mathbb{R}^{B \times H \times S_q \times S_k}$$

Both $S_q$ and $S_k$ must be declared as dynamic axes for the exported
graph to accept variable-length inputs.

In [ ]:
import torch
import torch.nn as nn
import tempfile, os, onnx

class MiniTransformerEncoder(nn.Module):
    """Minimal transformer encoder for demonstrating NLP export."""
    def __init__(self, vocab_size=1000, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = self.encoder(x)
        return self.classifier(x[:, 0, :])

nlp_model = MiniTransformerEncoder()
nlp_model.eval()

dummy_ids = torch.randint(0, 1000, (2, 16))  # batch=2, seq=16

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "mini_transformer.onnx")
    torch.onnx.export(
        nlp_model, dummy_ids, path,
        opset_version=17,
        input_names=["input_ids"],
        output_names=["logits"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq_len"},
            "logits":    {0: "batch"},
        },
    )
    m = onnx.load(path)
    onnx.checker.check_model(m)

    print(f"NLP model exported: {len(m.graph.node)} nodes")
    print(f"Unique ops: {sorted(set(n.op_type for n in m.graph.node))}")
    for inp in m.graph.input:
        dims = [d.dim_param or str(d.dim_value)
                for d in inp.type.tensor_type.shape.dim]
        print(f"Input '{inp.name}': [{', '.join(dims)}]")

<a id='section-12'></a>
## Section 12: Key Takeaways

### Export Checklist

```
┌─────────────────────────────────────────────────────────────────┐
│              PyTorch → ONNX EXPORT CHECKLIST                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  □  1. Call model.eval() before export                          │
│  □  2. Create dummy input matching production dtype and rank    │
│  □  3. Set input_names and output_names for graph clarity       │
│  □  4. Declare dynamic_axes for all variable dimensions         │
│  □  5. Use highest opset_version your runtime supports          │
│  □  6. Keep do_constant_folding=True (default)                  │
│  □  7. Run onnx.checker.check_model() on the result             │
│  □  8. Verify numerical parity (max abs diff < 1e-5)            │
│  □  9. Test with multiple batch sizes and input shapes          │
│  □ 10. Include parity test in CI pipeline                       │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Core Concepts Summary

| Concept | Key Insight |
|---------|------------|
| **Tracing** | Records ops on concrete input → straight-line graph; fails silently on data-dependent branches |
| **Scripting** | Analyzes Python source → preserves control flow; requires TorchScript-compatible code |
| **IR Lowering** | Maps TorchScript `aten::*` ops to ONNX `NodeProto` objects; higher opsets = simpler graphs |
| **Dynamic Axes** | Mark variable dimensions as symbolic; without them, shapes become frozen constants |
| **Constant Folding** | Pre-computes static subgraphs (e.g., fuses BatchNorm into Conv affine); reduces nodes |
| **Parity Checking** | Compare PyTorch vs ORT outputs; $\max |\Delta| < 10^{-5}$ for float32 |